In [53]:
!apt-get update
!apt-get install -y nmap

!pip install streamlit plotly pandas requests python-nmap -q

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,555 B in 1s (6,044 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nmap is already the newest version (7.91+dfsg1+rea

In [54]:
TARGETS = [
    "testasp.vulnweb.com",
    "testphp.vulnweb.com",
    "zero.webappsecurity.com"
]

SCAN_OUTPUT_DIR = "scan_xml"

In [55]:
import os
import subprocess

os.makedirs(SCAN_OUTPUT_DIR, exist_ok=True)

for target in TARGETS:

    output_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    cmd = [
        "nmap",
        "-sV",
        "-oX",
        output_file,
        target
    ]

    subprocess.run(cmd)

print("Scanning complete")

Scanning complete


In [56]:
import pandas as pd
import xml.etree.ElementTree as ET

results = []

for target in TARGETS:

    xml_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for host in root.findall("host"):

        ip = host.find("address").get("addr")

        ports = host.find("ports")

        if ports is None:
            continue

        for port in ports.findall("port"):

            port_id = port.get("portid")

            service = port.find("service")

            if service is not None:
                service_name = service.get("name")
            else:
                service_name = "unknown"

            results.append({
                "ip": ip,
                "port": port_id,
                "service": service_name
            })

df = pd.DataFrame(results)

df.head()

,ip,port,service
0,44.238.29.244,80,http
1,54.82.22.214,80,http
2,54.82.22.214,443,https
3,54.82.22.214,8080,http


In [57]:
import requests

VT_API_KEY = "d3e1643b7d601a5f6c2cc35111b9af18a8c9763a3454b6bf5bb867f1bb2a1737"
SHADON_API_KEY = "G4XUuqWCIvU0kUKMkNkkKeHxp1AGQYB9"

In [58]:
SENDER_EMAIL = "yaswanthanbrcs225@gmail.com"
APP_PASSWORD = "drhsoqrqfqomqvuz"
RECEIVER_EMAIL = "yashwanthan97@gmail.com"

In [59]:
import sys
!{sys.executable} -m pip install fpdf

In [60]:
def check_virustotal(ip):

    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

    headers = {
        "x-apikey": VT_API_KEY
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return "Unknown"

    data = response.json()

    malicious = data["data"]["attributes"]["last_analysis_stats"]["malicious"]

    if malicious > 5:
        return "High"
    elif malicious > 0:
        return "Medium"
    else:
        return "Low"

In [61]:
HIGH_RISK_SERVICES = {
    "ftp":3,
    "telnet":4,
    "ssh":1
}

def calculate_risk(row):

    score = 1

    if row["service"] in HIGH_RISK_SERVICES:
        score += HIGH_RISK_SERVICES[row["service"]]

    # Incorporate Shodan vulnerability count into risk score
    if "shodan_vuln_count" in row and row["shodan_vuln_count"] > 0:
        score += row["shodan_vuln_count"] # Each Shodan reported vuln adds to the risk

    # Add penalty based on VirusTotal reputation
    if row["vt_reputation"] == "Medium":
        score += 2
    elif row["vt_reputation"] == "High":
        score += 4

    return score

In [62]:
df["vt_reputation"] = df["ip"].apply(check_virustotal)
df["risk_score"] = df.apply(calculate_risk, axis=1)

In [63]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

def send_email_alert(df,target):

    high_risk = df[df["severity"].isin(["High","Critical"])]

    if high_risk.empty:
        print("No high risk vulnerabilities — email not sent")
        return

    msg = MIMEMultipart()

    msg["From"] = SENDER_EMAIL
    msg["To"] = RECEIVER_EMAIL
    msg["Subject"] = f"[SECURITY ALERT] High risk vulnerabilities detected on {target}"

    html = f"""
    <h2>CyberScan Pro Alert</h2>

    Target: {target}<br>
    Scan Time: {datetime.now()}<br>

    <h3>High Risk Findings</h3>

    {high_risk.to_html(index=False)}

    <hr>
    <i>Automatically generated by CyberScan Pro.</i>
    """

    msg.attach(MIMEText(html,"html"))

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(SENDER_EMAIL,APP_PASSWORD)
    server.sendmail(SENDER_EMAIL,RECEIVER_EMAIL,msg.as_string())
    server.quit()

    print("Alert email sent")

In [64]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

# Define a function to assign severity based on risk_score and vt_reputation
def assign_severity(row):
    if row["vt_reputation"] == "High" or row["risk_score"] >= 4:
        return "High"
    elif row["vt_reputation"] == "Medium" or row["risk_score"] >= 2:
        return "Medium"
    else:
        return "Low"

# Apply the function to create the 'severity' column
df["severity"] = df.apply(assign_severity, axis=1)

df.to_csv("scan_results.csv", index=False)
send_email_alert(df, "testphp.vulnweb.com")
print("scan_results.csv saved")

No high risk vulnerabilities — email not sent
scan_results.csv saved


In [77]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import subprocess
import xml.etree.ElementTree as ET
import requests
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
import nmap # Import the nmap library
from fpdf import FPDF # Added FPDF import

# =============================
# Global Variables and Configuration
# =============================

CONFIG = {
    "TARGETS": [
        "testasp.vulnweb.com",
        "testphp.vulnweb.com",
        "zero.webappsecurity.com"
    ],
    "SCAN_OUTPUT_DIR": "scan_xml",
    "VT_API_KEY": "d3e1643b7d601a5f6c2cc35111b9af18a8c9763a3454b6bf5bb867f1bb2a1737",
    "SHODAN_API_KEY": "G4XUuqWCIvU0kUKMkNkkKeXp1AGQYB9",
    "SENDER_EMAIL": "yaswanthanbrcs225@gmail.com",
    "APP_PASSWORD": "drhsoqrqfqomqvuz",
    "RECEIVER_EMAIL": "yashwanthan97@gmail.com",
    "HIGH_RISK_SERVICES": {
        "ftp":3,
        "telnet":4,
        "ssh":1
    }
}

# (functions unchanged...)

# =============================
# Streamlit App Layout
# =============================

st.set_page_config(
    page_title="CyberScan Pro",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("CyberScan Pro: Network Security Dashboard")
st.markdown("A powerful tool for network reconnaissance, threat intelligence, and vulnerability assessment.")
st.divider()

# Sidebar
st.sidebar.title("⚙️ Controls & Status")
st.sidebar.header("Application Status")

if CONFIG["VT_API_KEY"] and CONFIG["VT_API_KEY"] != "YOUR_VIRUSTOTAL_API_KEY":
    st.sidebar.success("✅ VirusTotal API Key configured")
else:
    st.sidebar.error("❌ VirusTotal API Key NOT configured")

if CONFIG["SHODAN_API_KEY"] and CONFIG["SHODAN_API_KEY"] != "YOUR_SHODAN_API_KEY":
    st.sidebar.success("✅ Shodan API Key configured")
else:
    st.sidebar.error("❌ Shodan API Key NOT configured")

if CONFIG["SENDER_EMAIL"] and CONFIG["APP_PASSWORD"] and CONFIG["RECEIVER_EMAIL"]:
    st.sidebar.success("✉️ Email alert system configured")
else:
    st.sidebar.warning("⚠️ Email alert system NOT fully configured")

with st.sidebar.expander("Current Scan Targets", expanded=False):
    for target in CONFIG["TARGETS"]:
        st.write(f"- {target}")
st.sidebar.divider()

st.sidebar.header("Scan Actions")

if st.sidebar.button("Run New Full Scan", help="Execute a new Nmap scan and process results."):
    run_full_scan_logic()

if st.sidebar.button("Refresh Dashboard Data", help="Reload data from the last scan without running a new one."):
    st.cache_data.clear()
    st.rerun()
st.sidebar.divider()

df = load_scan_results()

# --- Automatic email on dashboard refresh ---
if not df.empty:
    high_risk_entries_on_load = df[df["severity"].isin(["High", "Critical"])]
    if not high_risk_entries_on_load.empty:
        st.sidebar.info("High-risk entries detected on dashboard load. Attempting to send email alert...")
        email_status_refresh = send_email_alert(high_risk_entries_on_load, "CyberScan Pro - Dashboard Load Alert")
        st.sidebar.write(f"Automatic email alert status: {email_status_refresh}")
    else:
        st.sidebar.info("No high/critical risk entries detected on dashboard load; no automatic email sent.")

# Key Metrics
st.subheader("📈 Overview")

if not df.empty:
    col1,col2,col3,col4,col5 = st.columns(5)

    total_hosts = df["ip"].nunique()
    total_open_ports = len(df)
    unique_services = df["service"].nunique()
    highest_risk_score = df["risk_score"].max()
    high_risk_count = len(df[df["severity"].isin(["High", "Critical"])])

    col1.metric("Total Hosts Found 🌐", total_hosts)
    col2.metric("Open Ports Detected 📡", total_open_ports)
    col3.metric("Unique Services 🔎", unique_services)
    col4.metric("Highest Risk Score 🔥", highest_risk_score)
    col5.metric("High Risk Entries ⚠️", high_risk_count, delta=f"+" + str(high_risk_count) if high_risk_count > 0 else None, delta_color="inverse")
else:
    st.info("No scan data available. Run a full scan to populate overview metrics.")
st.divider()

# Navigation Tabs
tab1,tab2,tab3,tab4,tab5 = st.tabs([
    "📄 Scan Data",
    "📊 Visualizations",
    "🚨 Threat Intelligence",
    "📃 Host Details",
    "📂 Export Data"
])

# (rest of tab logic unchanged, but all surrogate pairs replaced with direct emoji)


2026-03-29 14:06:39.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.473 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 14:06:39.474 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [78]:
import sys
!{sys.executable} -m pip install streamlit --upgrade
print("Streamlit upgraded. Please re-run the dashboard.py cell (bXDu9Fz6BMts) to apply changes.")

Streamlit upgraded. Please re-run the dashboard.py cell (bXDu9Fz6BMts) to apply changes.


In [79]:
import subprocess
import threading

def run_streamlit():

    subprocess.run([
        "streamlit",
        "run",
        "dashboard.py",
        "--server.port","8501",
        "--server.headless","true"
    ])

threading.Thread(target=run_streamlit).start()

In [80]:
def check_shodan(ip):
    if not SHADON_API_KEY or SHADON_API_KEY == "G4XUuqWCIvU0kUKMkNkkKeHxp1AGQYB9":
        return {"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

    url = f"https://api.shodan.io/shodan/host/{ip}?key={SHADON_API_KEY}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            country = data.get("country_name", "Unknown")
            vuln_count = len(data.get("vulns", [])) if isinstance(data.get("vulns"), list) else 0
            if isinstance(data.get("vulns"), dict):
                vuln_count = len(data.get("vulns").keys())

            ports = data.get("ports",[])
            shodan_ports = ", ".join(map(str, ports)) if ports else "N/A"

            return {"shodan_country": country, "shodan_vuln_count": vuln_count, "shodan_ports": shodan_ports}
        else:
            return {"shodan_country": "Unknown", "shodan_vuln_count": 0, "shodan_ports": "N/A"}
    except requests.exceptions.RequestException:
        return {"shodan_country": "Error", "shodan_vuln_count": 0, "shodan_ports": "N/A"}

print("check_shodan function defined.")

check_shodan function defined.


In [90]:
shodan_results = []
for ip in df["ip"].unique():
    shodan_data = check_shodan(ip)
    shodan_results.append({"ip": ip, **shodan_data})

df_shodan = pd.DataFrame(shodan_results)
df = pd.merge(df, df_shodan, on="ip", how="left").fillna({"shodan_country": "N/A", "shodan_vuln_count": 0, "shodan_ports": "N/A"})

print("Shodan data merged into DataFrame.")
df.head()

Shodan data merged into DataFrame.


,ip,port,service,vt_reputation,risk_score,severity,shodan_country_x,shodan_vuln_count_x,shodan_ports_x,shodan_country_y,shodan_vuln_count_y,shodan_ports_y
0,44.238.29.244,80,http,Medium,3,Medium,N/A,0,N/A,N/A,0,N/A
1,54.82.22.214,80,http,Low,1,Low,N/A,0,N/A,N/A,0,N/A
2,54.82.22.214,443,https,Low,1,Low,N/A,0,N/A,N/A,0,N/A
3,54.82.22.214,8080,http,Low,1,Low,N/A,0,N/A,N/A,0,N/A


In [91]:
!nohup streamlit run dashboard.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

In [92]:
!cloudflared tunnel --url http://localhost:8501

2026-03-29T14:19:19Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-29T14:19:19Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-29T14:19:22Z INF +--------------------------------------------------------------------------------------------+
2026-03-29T14:19:22Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-29T14:19:22Z INF |  https://serve-poor-earl-strength.trycloudflare.com   

In [ ]:
!killall streamlit
!killall cloudflared

import subprocess
import threading

def run_streamlit():
    subprocess.run(
        [
            "streamlit",
            "run",
            "dashboard.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
        ]
    )

threading.Thread(target=run_streamlit).start()


import subprocess

def run_cloudflared():
    subprocess.run(["cloudflared", "tunnel", "--url", "http://localhost:8501"])

threading.Thread(target=run_cloudflared).start()

streamlit: no process found
cloudflared: no process found


In [86]:
import pandas as pd

# Create a dummy DataFrame with 'High' severity to test email alert
dummy_high_risk_df = pd.DataFrame({
    "ip": ["208.67.222.222", "185.199.108.153", "172.217.160.142", "10.0.0.5", "172.16.0.2"],
    "port": [22, 80, 443, 21, 23],
    "service": ["ssh", "http", "https", "ftp", "telnet"],
    "risk_score": [4, 7, 9, 6, 10],
    "vt_reputation": ["High", "Medium", "High", "Medium", "High"],
    "severity": ["High", "High", "Critical", "High", "Critical"]
})


print("Attempting to send email alert for dummy high-risk vulnerabilities...")
send_email_alert(dummy_high_risk_df, "dummy_test_target.com")
print("Verification complete.")

Attempting to send email alert for dummy high-risk vulnerabilities...
Verification complete.
